# Boosted Decision Tree

In [9]:
import pandas as pd
import numpy as np
import re
import math
import time
import warnings
from datetime import timedelta
from tabulate import tabulate
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score
# Nascondo i warning
warnings.filterwarnings('ignore')
# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training


In [ ]:
def training(file_path, csv_name):

    df = pd.read_csv(file_path)

    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']

    # Vado a tenere i target che esistono del csv
    available_targets = [c for c in original_target_list if c in df.columns]

    # tengo solo i casi con tutti i target disponibili
    df_validi = df.dropna(subset=available_targets).copy()

    final_target_list = []
    # Binarizzazione dei marker
    if 'PR [SII]' in available_targets:
            df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
            final_target_list.append('PR_class')

    if 'ER [SII]' in available_targets:
        df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
        final_target_list.append('ER_class')

    if 'KI67 [%]' in available_targets:
        df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
        final_target_list.append('KI67_class')

    if 'HER2 [SII]' in available_targets:
        df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)
        final_target_list.append('HER2_class')



    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign',
                        'GRADE', 'isTN', 'Breast'] + available_targets + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    features = features.fillna(features.mean())
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    # 4 fold per paziente
    cv = GroupKFold(n_splits=5)

    base_model = HistGradientBoostingClassifier(
        random_state=42,
        class_weight='balanced',
        early_stopping=False,
        l2_regularization=0.0,
        min_samples_leaf=10
    )

    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
        'estimator__learning_rate': [0.05, 0.1, 1],
        'estimator__max_iter': [100, 200],
        'estimator__max_depth': [3, 5]
    }

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        scores = []
        for i in range(y_true.shape[1]):
            scores.append(
                f1_score(y_true[:, i], y_pred[:, i],
                         average='macro', zero_division=0)
            )
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search HistGradientBoosting (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,
        verbose=1,
        refit=True,
        error_score='raise'
    )

    grid_search.fit(features, target, groups=groups)

    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    clean_best_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}

    final_params = {
        'random_state': 42,
        'class_weight': 'balanced',
        'early_stopping': False,
        'l2_regularization': 0.0,
        'min_samples_leaf': 10,
        **clean_best_params
    }

    # metriche per FOLD e per LABEL
    fold_reports = []

    for train_idx, test_idx in cv.split(features, target, groups):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        model_clone = MultiOutputClassifier(HistGradientBoostingClassifier(**final_params))
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)

        fold_metrics = {}

        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i]
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            # AUC
            unique_classes = np.unique(y_true_i)
            if len(unique_classes) < 2:
                auc_val = np.nan
            else:
                try:
                    if y_proba_list[i].shape[1] == 2:
                        auc_val = roc_auc_score(y_true_i, y_proba_list[i][:, 1])
                    else:
                        auc_val = 0.5
                except ValueError:
                    auc_val = np.nan

            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'auc': auc_val
            }

        fold_reports.append(fold_metrics)

    final_result = [{
        **clean_best_params,
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports,
        'targets_used': final_target_list
    }]
    
    return final_result


# Vado a stampare gli output in una maniera piú leggibile

In [11]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 20 + "Migliori metrice per ogni fold")
    print("=" * 80)

    for dataset_name, metrics_list in results_per_dataset.items():
        if not metrics_list:
            continue

        best_result = metrics_list[0]     # modello migliore per il dataset
        fold_reports = best_result["fold_reports"]


        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        target_names =  best_result.get("targets_used", [])

        for target in target_names:

            # --- LISTE DI VALORI SUI FOLD ---
            f1_list  = np.array([fold[target]["f1"] for fold in fold_reports])
            acc_list = np.array([fold[target]["accuracy"] for fold in fold_reports])
            auc_list = np.array([fold[target]["auc"] for fold in fold_reports], dtype=float)

            # --- BEST VALUES ---
            best_f1  = np.max(f1_list)
            best_acc = np.max(acc_list)

            # Per AUC rimuovo eventuali NaN
            valid_auc = ~np.isnan(auc_list)
            best_auc  = np.max(auc_list[valid_auc]) if valid_auc.any() else np.nan

            # --- STD DEV ---
            f1_std  = np.std(f1_list)
            acc_std = np.std(acc_list)
            auc_std = np.std(auc_list[valid_auc]) if valid_auc.any() else np.nan

            # --- STAMPO RISULTATI ---
            print(f"\nTarget: {target}")
            print(f"  F1-score     = {best_f1:.3f} " +" ± "+f" {f1_std:.3f}")
            print(f"  Accuracy     = {best_acc:.3f} " +" ± "+f" {acc_std:.3f}")
            print(f"  AUC          = {best_auc:.3f} " +" ± "+f" {auc_std:.3f}" if not np.isnan(best_auc) else
                  f"  AUC          = NaN           " +" ± "+f" NaN")


# Lettura dei file

In [12]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)




end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search HistGradientBoosting (GRID MINIMAL: 12 combinazioni) per: ambl_lesions
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[duke_lesions] ATTENZIONE: il target HER2_class ha una sola classe globale 0, lo escludo.

Inizio Grid Search HistGradientBoosting (GRID MINIMAL: 12 combinazioni) per: duke_lesions
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Inizio Grid Search HistGradientBoosting (GRID MINIMAL: 12 combinazioni) per: t2_medsam
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Inizio Grid Search HistGradientBoosting (GRID MINIMAL: 12 combinazioni) per: t2_preprocessed
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Inizio Grid Search HistGradientBoosting (GRID MINIMAL: 12 combinazioni) per: t2_original
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Inizio Grid Search HistGradientBoosting (GRID MINIMAL: 12 combinazioni) per: medsam_dynamic
Fitting 5 folds for each of 12 candidates, totalling 60 fi